# Flickr30k T1→S1 Hybrid KD Closeout

Đóng nhánh Hybrid sau tổng hợp ba seed. Notebook chỉ ghi nhận bằng chứng
Validation hậu nghiệm và không thay đổi khóa triển khai T3→S3 hiện hành.

## Kaggle input bắt buộc — đúng 3 input

1. Output `F30K_T1S1_Hybrid_Protocol`.
2. Output `F30K_T1S1_Hybrid_Aggregate`.
3. Output `F30K_Final_Audit`.

Runtime: CPU/Accelerator None, Internet Off, khoảng 1–2 phút.

## Cell 1 — Scientific closeout

In [1]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_T1S1_Hybrid_Closeout"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def write_json(path, payload):
    temporary = Path(path).with_suffix(Path(path).suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

protocol_path = unique_named(
    "F30K_T1S1_Hybrid_Protocol_RESULT.json"
)
aggregate_path = unique_named(
    "F30K_T1S1_Hybrid_Aggregate_RESULT.json"
)
final_audit_path = unique_named("F30K_Final_Audit_RESULT.json")
protocol, aggregate, final_audit = map(
    read_json, (protocol_path, aggregate_path, final_audit_path)
)
assert protocol["status"] == "locked_validation_only"
assert protocol["test_evaluation_authorized"] is False
assert aggregate["status"] == "complete"
assert aggregate["study_status"] == "post_hoc_validation_only"
assert aggregate["test_set_used"] is False
assert aggregate["test_evaluation_authorized"] is False
assert aggregate["seeds"] == [42, 123, 2026]
assert final_audit["status"] in {"complete", "passed"}

positive = bool(aggregate["consistent_positive_delta"])
conclusion = (
    "PROMISING_ON_VALIDATION_REQUIRES_NEW_INDEPENDENT_EVALUATION"
    if positive else aggregate["decision"]
)
closeout = {
    "schema_version": "f30k_t1s1_hybrid_closeout_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "method": aggregate["method"],
    "study_status": "post_hoc_validation_only",
    "seeds": aggregate["seeds"],
    "validation_summary": aggregate["summary"],
    "paired_bootstrap": aggregate["combined_paired_bootstrap"],
    "conclusion": conclusion,
    "existing_locked_deployment_model_unchanged": True,
    "existing_locked_model": "T3S3 Direct Logit-KD INT8",
    "test_set_status": "previously_opened_for_locked_T3S3_only",
    "test_set_used_by_followup": False,
    "test_evaluation_authorized": False,
    "onnx_or_int8_authorized_for_hybrid": False,
    "next_required_if_promising": (
        "Register a new independent unseen evaluation cohort/dataset before "
        "any confirmatory claim or deployment replacement."
    ),
    "source_sha256": {
        "protocol": sha256(protocol_path),
        "aggregate": sha256(aggregate_path),
        "final_audit": sha256(final_audit_path),
    },
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, closeout)
report = [
    "# T1→S1 Hybrid KD — scientific closeout",
    "",
    f"- Status: `{closeout['study_status']}`",
    f"- Conclusion: `{conclusion}`",
    f"- Mean Validation CIDEr-D delta: "
    f"`{aggregate['summary']['CIDEr_D']['delta_mean']:+.6f}`",
    f"- Combined paired-bootstrap CI95: "
    f"`[{aggregate['combined_paired_bootstrap']['ci95_lower']:+.6f}; "
    f"{aggregate['combined_paired_bootstrap']['ci95_upper']:+.6f}]`",
    "- Existing T3→S3 deployment lock: `UNCHANGED`",
    "- Old held-out Test: `NOT USED / NOT AUTHORIZED`",
    "- Hybrid ONNX/INT8: `NOT AUTHORIZED BY THIS EXPLORATORY STUDY`",
]
report_path = WORK_ROOT / "REPORT.md"
report_path.write_text("\n".join(report) + "\n", encoding="utf-8")
artifacts = []
for path in (result_path, report_path):
    artifacts.append({
        "relative_path": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t1s1_hybrid_closeout_manifest_v1",
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "artifacts": artifacts,
    "test_set_status": closeout["test_set_status"],
    "test_set_used_by_followup": False,
})

print("Status                   : COMPLETE")
print("Conclusion               :", conclusion)
print("Mean Validation CIDEr-D Δ:",
      aggregate["summary"]["CIDEr_D"]["delta_mean"])
print("Existing deployment lock : UNCHANGED")
print("Old held-out Test         : NOT USED / NOT AUTHORIZED")

Status                   : COMPLETE
Conclusion               : STOP_VALIDATION_NO_GAIN
Mean Validation CIDEr-D Δ: -0.09235649791067067
Existing deployment lock : UNCHANGED
Old held-out Test         : NOT USED / NOT AUTHORIZED
